# Create Dataset Samples

This notebook filters raw Orbis parquet files by a country prefix in `bvd_id_number` and exports CSV files for downstream work.

The cells below configure the source and output paths, inspect country match counts, export country-filtered samples, create two 2014 financial extracts, and inspect the resulting files.


In [ ]:
# Import path utilities and Polars for reading and filtering parquet data.
from pathlib import Path
import polars as pl
from datetime import datetime


## Configuration

Set the input directories, output directories, country filter, and row cap here. The same `country_iso2` and `max_rows` values are reused throughout the notebook.


In [ ]:
# Source directories
orbis_data_dir: Path = Path("./orbis_data/Orbis_Academics_Quarterly")

# Output directories for the sampled CSV exports
orbis_sample_dir: Path = Path("orbis_sample/")

# Filter settings
# Use the two-letter BvD identifier prefix and set the per-file sample cap.
country_iso2 = "FR"
max_rows = int(1e5)
max_rows = int(2.4e7)

# Filter by target_date and child_name
target_date = datetime(2014, 12, 31, 0, 0, 0)


## Quarterly Orbis Files

First, collect the parquet files that belong to the quarterly Orbis extract. macOS metadata files such as `.DS_Store` are excluded so the later parquet scan only sees real data files.


In [ ]:
# Collect the quarterly parquet dataset directories, excluding macOS metadata.
path_list = [
    filepath
    for filepath in orbis_data_dir.iterdir()
    if filepath.name != ".DS_Store"
]


### Inspect Match Counts

This pass stays lazy in Polars. It reads file metadata to count total rows, applies the country filter, and then counts the matching rows so you can see sample sizes before exporting anything.


In [ ]:
# Count total rows and country-matching rows in each parquet dataset.
for directory_name in path_list:
    # Create a lazy parquet scan so filtering and counting can be pushed down.
    lp = pl.scan_parquet(directory_name)

    # Count all rows in the source file.
    raw_file_length = lp.select(pl.len()).collect().item()

    # Keep only rows whose BvD identifier starts with the requested ISO2 code.
    filtered_lp = lp.filter(pl.col("bvd_id_number").str.starts_with(country_iso2))

    # Count how many rows survive the country filter.
    total_matches = filtered_lp.select(pl.len()).collect().item()

    print(f"File: {Path(directory_name).name}")
    print(f"  - Total rows in file: {raw_file_length:,}")
    print(f"  - Matches for {country_iso2}: {total_matches:,}")

    # The export step is intentionally separate so counts can be reviewed first.
    # df = filtered_lp.limit(max_rows).collect()
    # df.write_csv(...)


### Export Quarterly Sample

This pass materialises the filtered rows and writes each parquet file to a CSV with the same base filename. The export is capped at `max_rows` per source file.


In [ ]:
# Export up to max_rows rows per dataset after filtering to the selected country.
for directory_name in path_list:
    path = Path(directory_name)
    child_name = path.name
    new_path = Path(orbis_sample_dir / child_name)

    df = (
        pl.scan_parquet(directory_name)
        # Restrict the extract to companies from the selected country.
        .filter(pl.col("bvd_id_number").str.starts_with(country_iso2))
        # Keep at most max_rows rows per parquet file.
        .limit(max_rows)
        # Execute the lazy query and bring the result into memory.
        .collect()
    )

    df.write_csv(f"{new_path}.csv")


In [ ]:
filter_date = True

if filter_date is True:

    # Export the global financials-and-ratios dataset for the target year-end.
    for directory_name in path_list:
        path = Path(directory_name)
        child_name = path.name
        new_path = Path(orbis_sample_dir / child_name)

        if child_name == "orbis_academics_quarterly_industry_global_financials_and_ratios":
            # Keep only the selected country and closing date, then write a CSV.
            df = (
                pl.scan_parquet(directory_name)
                .filter(pl.col("bvd_id_number").str.starts_with(country_iso2))
                .filter(pl.col("closing_date") == pl.lit(target_date))
                .collect()
            )

            df.write_csv(f"{new_path}_2014.csv")
            break

In [ ]:
# Export the detailed-format EUR dataset for the same country and year-end.
detailed_format = True

if detailed_format is True:

    for directory_name in path_list:
        path = Path(directory_name)
        child_name = path.name
        new_path = Path(orbis_sample_dir / child_name)

        if child_name == "orbis_academics_quarterly_detailed_format_industries_eur":
            # Keep only the selected country and closing date, then write a CSV.
            df = (
                pl.scan_parquet(directory_name)
                .filter(pl.col("bvd_id_number").str.starts_with(country_iso2))
                .filter(pl.col("closing_date") == pl.lit(target_date))
                .collect()
            )

            df.write_csv(f"{new_path}_2014.csv")
            break


In [ ]:
# Export country-matching rows from the industry classifications dataset.
industry_classification = True

if industry_classification is True:

    for directory_name in path_list:
        path = Path(directory_name)
        child_name = path.name
        new_path = Path(orbis_sample_dir / child_name)

        if child_name == "orbis_academics_quarterly_industry_classifications":
            # Keep only the selected country before writing the classification CSV.
            df = (
                pl.scan_parquet(directory_name)
                .filter(pl.col("bvd_id_number").str.starts_with(country_iso2))
                .collect()
            )

            df.write_csv(f"{new_path}.csv")
            break

In [ ]:
# Read the global financials extract and print its column names for inspection.
inspect_data = True
if inspect_data is True:
    inspection_path = Path(orbis_sample_dir / "orbis_academics_quarterly_industry_global_financials_and_ratios_2014.csv")
    df = pl.read_csv(inspection_path, ignore_errors=True)
    print(df.columns)

In [ ]:
# List files in the sample directory to confirm which CSVs were created.
show_files = True
if show_files is True:
    sample_dir = Path("/Users/andone/Documents/Orbis/orbis_sample")
    for file_path in sorted(sample_dir.iterdir()):
        if file_path.is_file():
            print(file_path.name)
